# F16 — Baseline DL ResNet-1D en Vast.ai

Entrena el baseline sobre los shards F15 ya cargados en la instancia. Este
cuaderno se ejecuta en la plantilla **PyTorch de Vast.ai** por SSH directo o
Jupyter de Vast; no usa Colab, Google Drive, Tailscale ni credenciales.

Primero ejecuta el preflight de `05_dl_colab.ipynb`. Conserva el conjunto
`test` sin usar: F16 selecciona el modelo solo con `val`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = Path(os.environ.get('ECG_REPO', '/workspace/ecg')).resolve()
DL = REPO / 'results' / 'dl'
RUNS = REPO / 'runs'
RUNS.mkdir(exist_ok=True)
assert len(list((DL / 'shards').glob('shard_*.npy'))) == 7, (
    'Faltan shards: ejecuta primero el preflight F15.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'numpy', 'pandas'],
               check=True)

import torch
assert torch.cuda.is_available(), 'F16 completo requiere CUDA.'
print('Repo:', REPO)
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__, '| CUDA:', torch.version.cuda)
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['df', '-h', str(REPO)], check=True)

## 1. Humo de GPU

El humo usa hasta 200 ventanas por clase, dos épocas y la misma tubería de
lectura/normalización. Debe terminar con `F16 HUMO VERDE` antes de gastar
una corrida completa.

In [ ]:
import time

humo_log = RUNS / 'f16_humo.log'
with humo_log.open('w') as log:
    proceso = subprocess.run(
        [sys.executable, 'scripts/entrenar_baseline_dl.py', '--humo'],
        cwd=REPO, stdout=log, stderr=subprocess.STDOUT, text=True)
assert proceso.returncode == 0, humo_log.read_text()
print(humo_log.read_text())

## 2. Entrenamiento completo

La salida queda localmente en `runs/f16_baseline/`: `mejor.pt`, `ultimo.pt`,
`historial.json` y `gate_f16.txt`. La instancia actual no tiene volumen
persistente; descarga estos archivos antes de destruirla o reciclarla.

La RTX 5060 Ti tiene 16 GB. `batch=64` es el punto de partida; si aparece
`CUDA out of memory`, baja a 32 y vuelve a iniciar una corrida nueva.

In [ ]:
SALIDA = RUNS / 'f16_baseline'
SALIDA.mkdir(exist_ok=True)
cmd = [
    sys.executable, 'scripts/entrenar_baseline_dl.py',
    '--epocas', '30', '--batch', '64', '--lr', '3e-4',
    '--salida', str(SALIDA),
]
print('Comando:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO, check=True)
print((SALIDA / 'gate_f16.txt').read_text())

## 3. Rescate de resultados

Desde el equipo local, descarga los artefactos antes de destruir la instancia:

```bash
scp -i ~/.ssh/keys/vast-ai/id_ed25519 -P <PUERTO> -r \
  root@<IP>:/workspace/ecg/runs/f16_baseline ./resultados_f16
```

Revisa `gate_f16.txt` y `historial.json`. No ejecutes evaluación sobre test
durante F16; esa evaluación pertenece a F18.